In [1]:
import requests
import re
import pandas as pd
import numpy as np
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss
import matplotlib.pyplot as plt
import os
import requests
import io

In [2]:
#Parameters
Iterations = 2048 # number of iterations for Logistic Regression
random = 32 # random state for Logistic Regression

In [ ]:
# Load Data
api_url = "https://api.github.com/repos/COMP3608-GROUP-9-PROJECT/COMP3608_PROJECT/contents/ProcessedData/processed_tournament_data.csv"
os.makedirs("processed", exist_ok=True)

response = requests.get(api_url)

if response.status_code == 200:
    file = response.json()
    download_url = file["download_url"]

    csv_data = requests.get(download_url).content
    tournament_data = pd.read_csv(io.BytesIO(csv_data))

    print("Loaded into DataFrame successfully")
else:
    raise Exception(f"Failed to access API: {response.status_code}")

TARGET = "Win"

FEATURES = [col for col in tournament_data.columns if col not in ["Win", "Season", "PointDifference", "Divison", "SeedDifference","RatingDifference"]]

x = tournament_data[FEATURES]
y = tournament_data[TARGET]

groups = tournament_data["Season"]
seasons = tournament_data["Season"].unique()

gkf = GroupKFold(n_splits=len(seasons))


Loaded into DataFrame successfully


In [6]:
model = LogisticRegression(max_iter=Iterations, random_state=random)

cv_results = []

for season_index, (train_index, test_index) in enumerate(gkf.split(x, y, groups)):
    test_season = groups.iloc[test_index].unique()[0]
    x_train, x_test = x.iloc[train_index], x.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model_fold = clone(model)
    model_fold.fit(x_train, y_train)

    y_pred = model_fold.predict(x_test)
    accuracy = accuracy_score(y_test, y_pred)
    y_pred_prob = model_fold.predict_proba(x_test)[:, 1]
    log_loss_score = log_loss(y_test, y_pred_prob)
    brier_score = brier_score_loss(y_test, y_pred_prob)

    cv_results.append({
        "Season": test_season,
        "Accuracy": accuracy,
        "LogLoss": log_loss_score,
        "BrierScore": brier_score
    })


df_cv_results = pd.DataFrame(cv_results)
df_cv_results = df_cv_results.sort_values("Season")

for _, row in df_cv_results.iterrows():
    print(f"Season {int(row['Season'])}: Accuracy = {row['Accuracy']:.3f}, Log Loss = {row['LogLoss']:.3f}, Brier Score = {row['BrierScore']:.3f}")

print(model.fit(x, y).score(x, y))

print(f"\nAverage Accuracy: {df_cv_results['Accuracy'].mean():.3f}")
print(f"Average Log Loss: {df_cv_results['LogLoss'].mean():.3f}")
print(f"Average Brier Score: {df_cv_results['BrierScore'].mean():.3f}")

Season 2003: Accuracy = 0.797, Log Loss = 0.546, Brier Score = 0.180
Season 2004: Accuracy = 0.688, Log Loss = 0.599, Brier Score = 0.210
Season 2005: Accuracy = 0.688, Log Loss = 0.611, Brier Score = 0.212
Season 2006: Accuracy = 0.734, Log Loss = 0.604, Brier Score = 0.207
Season 2007: Accuracy = 0.656, Log Loss = 0.597, Brier Score = 0.207
Season 2008: Accuracy = 0.734, Log Loss = 0.543, Brier Score = 0.179
Season 2009: Accuracy = 0.703, Log Loss = 0.596, Brier Score = 0.205
Season 2010: Accuracy = 0.654, Log Loss = 0.610, Brier Score = 0.212
Season 2011: Accuracy = 0.700, Log Loss = 0.555, Brier Score = 0.191
Season 2012: Accuracy = 0.669, Log Loss = 0.570, Brier Score = 0.198
Season 2013: Accuracy = 0.669, Log Loss = 0.589, Brier Score = 0.204
Season 2014: Accuracy = 0.646, Log Loss = 0.564, Brier Score = 0.196
Season 2015: Accuracy = 0.738, Log Loss = 0.525, Brier Score = 0.177
Season 2016: Accuracy = 0.662, Log Loss = 0.634, Brier Score = 0.223
Season 2017: Accuracy = 0.692, Log

In [ ]:
#Plot results to graph
plt.figure(figsize=(10, 5))
plt.plot(df_cv_results['Season'], df_cv_results['Accuracy'], marker='o', label='Accuracy')
plt.plot(df_cv_results['Season'], df_cv_results['LogLoss'], marker='o', label='Log Loss')
plt.xlabel("Season")
plt.ylabel("Score")
plt.title("Logistic Regression Predictions on Selected Features")
plt.legend()
plt.show()

In [ ]:
api_url = "https://api.github.com/repos/COMP3608-GROUP-9-PROJECT/COMP3608_PROJECT/contents/ProcessedData/Sample_Submission.csv"
os.makedirs("processed", exist_ok=True)

response = requests.get(api_url)

if response.status_code == 200:
    file = response.json()
    download_url = file["download_url"]

    csv_data = requests.get(download_url).content
    sample_submission = pd.read_csv(io.BytesIO(csv_data))

    print("Loaded into DataFrame successfully")
else:
    raise Exception(f"Failed to access API: {response.status_code}")


In [ ]:
x_new = sample_submission[FEATURES]
sample_submission["Pred"] = model.predict_proba(x_new)[:, 1]
sample_submission[['ID', 'Pred']].to_csv("LogisticRegressionPredictions.csv", index=False)